# M52 · Synthetic seismograms: moment tensors, radiation, whole-space and layered-Earth synthetics

**Week 6 · about 10 h** · before this: M10, M41, M50

> **한국어 요약** 이 모듈에서는 지진 하나를 점 진원으로 보고 그 지진이 만드는 땅의 움직임, 즉 합성 지진 기록을 직접 계산합니다. 먼저 주향·경사·미끄럼각과 지진 모멘트로 모멘트 텐서를 만들고(북-동-아래 좌표와 위-남-동 좌표), 지진 모멘트와 모멘트 규모를 서로 바꾸며, P파·SV파·SH파의 방사 패턴을 정답을 아는 가상의 단층으로 검증합니다. 다음으로 무한 균질 매질에서 원거리장 변위를 계산하는 함수를 작성하고, 별도의 Python 3.9 환경에서 돌아가는 PyFK(주파수-파수 방법)의 결과와 여러 거리에서 비교해 근거리장의 효과를 측정합니다. 동료가 PyFK를 부른 셀을 그대로 실행해 본 뒤, PyFK가 원천의 첫 번째 값으로 무엇을 받고 어떤 물리량을 어떤 단위로 돌려주는지 먼저 예측하고, PyFK의 코드와 직접 잰 값으로 확인합니다. 마지막으로 KIGAM이 발표한 2019년 상주 지진의 단층면해와 kim1983 속도 모델로 합성 기록을 만들어, 계기 응답을 제거하고 같은 대역으로 거른 실제 기록과 파형 모양과 진폭 비를 비교하고, 진폭 비를 모멘트 규모의 차이로 바꾸어 보며, 그대로 다시 실행한 뒤 변수 하나를 바꾸어 그 차이를 설명합니다.

**You will**

- Build the moment tensor of a double couple from strike, dip, rake and seismic moment in the north-east-down and up-south-east frames, and convert between seismic moment and moment magnitude
- Compute the far-field P, SV and SH radiation pattern of a fault and test it against cases whose answer is known by construction
- Write a whole-space far-field point-source synthetic and compare it with PyFK at several distances, measuring where the near field matters
- Find out from PyFK's code and from a measurement of your own what PyFK expects as its source and what kind of motion it returns, and test a colleague's call against what you find
- Compare layered-model synthetics of the 2019 Sangju earthquake with response-corrected, band-passed records (shape, time shift and amplitude ratio), express the amplitude ratio as a change of moment magnitude, rerun them unchanged, change one parameter and explain the difference

**New words:** synthetic seismogram (합성 지진 기록, 합성 파형), point source (점 진원), moment tensor (모멘트 텐서), double couple (이중 우력, 더블 커플), seismic moment (M0; 지진 모멘트), moment magnitude (Mw; 모멘트 규모), dyne-centimetre (dyne-cm; 다인-센티미터), strike (주향) · [all terms](../../../docs/glossary.md)

**Optional reading:** Aki & Richards (2002), chapter 4 (the representation theorem, Box 4.4 on the moment tensor of a fault, the far-field terms of equation 4.29 and the radiation patterns of equations 4.84-4.89); Jost & Herrmann (1989), the sections on the moment tensor of a double couple and its coordinate systems; Shearer (2019), the chapter on earthquakes (moment tensors, radiation patterns and moment magnitude); Zhu & Rivera (2002), the introduction and the description of the method (what the FK method computes)

Log any AI help in `work/ai_log/M52.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M52.yaml](../../../.sgtl/criteria/M52.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
BAND = "HH"                               # broadband channels
VELOCITY_MODEL = "kim1983"                # layered model of the Sangju synthetics (Part 6)
CHANGED_MODEL = "kim2011"                 # Part 6: the one parameter you change in the second run
KIGAM_PLANE_A = (288.0, 89.0, 7.0)        # strike, dip, rake (degrees); KIGAM (2021), report KEMS-2019-062
KIGAM_PLANE_B = (198.0, 83.0, 179.0)      # the other nodal plane of the same report
KIGAM_MW = 3.8                            # moment magnitude of the same report
KIGAM_DEPTH_KM = 12.8                     # focal depth of the same report (km)
FIT_STATIONS = ["KG.MGB", "KS.YCHB", "KS.OKCB", "KS.CHJ2", "KS.EUSB", "KS.YOJB"]   # Part 6 (15-60 km)
PRE_FILT_HZ = (0.1, 0.2, 30.0, 40.0)      # response removal (M10): frequency taper (Hz)
FIT_BAND_HZ = (0.5, 2.0)                  # band-pass of records and synthetics in Part 6 (Hz)
MAX_LAG_S = 2.0                           # largest time shift searched in Part 6 (s)
FK_DT_S, FK_NPT = 0.05, 2048              # PyFK samples of Part 6: interval (s) and number (102 s)
STF_DURATION_S = 0.4                      # Part 6: duration of PyFK's trapezoid source time function (s)
TEST_MECHANISM = (40.0, 65.0, -20.0)      # Parts 4-5: a made-up fault (strike, dip, rake in degrees)
TEST_MW = 3.0                             # Parts 4-5: its moment magnitude
WS_VP, WS_VS, WS_RHO = 6.0, 3.5, 2700.0   # Parts 4-5: the uniform rock (km/s, km/s, kg/m3)
WS_SOURCE_KM, WS_RECEIVER_KM = 100.0, 90.0   # Part 5: depths of the source and of the buried receivers (km)
WS_DISTANCES_KM = [5.0, 15.0, 40.0, 80.0]    # Part 5: horizontal distances of the receivers (km)
WS_AZIMUTHS_DEG = [20.0, 110.0, 200.0, 290.0]  # Part 5: their azimuths from the source (degrees)
WS_DT_S, WS_NPT = 0.02, 2048              # Part 5: PyFK samples (s, number)
WS_STF_S = 1.0                            # Parts 4-5: duration of the smooth (Hann) moment-rate pulse (s)
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files (PyFK runs, tables) go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M52")

import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Trace, UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth
from sgtl_tutorial import pyfk_runner as fk
from sgtl_tutorial._paths import short_path

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M52-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up faults, rocks and records whose answers are
known from how they were built (`st.reroll("M52-T1")` draws new ones); task 6 also reruns your function on your
own Sangju files and never prints a value of its own for them.

The module climbs a ladder in three steps. **R1**: write the core functions (a moment tensor, a magnitude, a
radiation pattern, a far-field seismogram) and test them on made-up faults with known answers. **R2**: compare
them with an independent program, PyFK, and then with real records of the 2019 Sangju earthquake. **R3**: run
the production tool, PyFK, for that earthquake: once, again unchanged, and once with one parameter changed.

Before Part 4, run `bin/sgtl-install pyfk` in a server terminal, read its plan, then repeat with `--apply`.
It creates the separate PyFK worker environment; keep this notebook on the **SGTL (Python 3.12)** kernel.
Part 6 uses the `sangju2019` records and station metadata from M03. The marked paste cells request your own
checked functions from M50 and M41, including any helpers they call.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M52.md`.

## Part 0 · What a synthetic seismogram is (about 20 min)

A **synthetic seismogram** (glossary `synthetic_seismogram`) is the ground motion that a model earthquake would
cause at a station in a model Earth. Three pieces make it:

1. the **source**: here a **point source** (glossary `point_source`), one point that slips, described by its
   **moment tensor** (glossary `moment_tensor`) and by how fast the slip grows with time, the **source time
   function** (glossary `source_time_function`);
2. the **medium**: a **velocity model** (glossary `velocity_model`) of speeds and densities;
3. the **path effect**, which the medium gives: the **Green's function** (glossary `greens_function`), the motion
   that an impulse at the source causes at the station.

For a uniform rock without boundaries, a **whole space** (glossary `whole_space`), Aki and Richards (2002,
chapter 4) write the motion as a formula: you will code its far-field part. For a layered crust there is no such
formula; PyFK (Xi et al.; Zhu and Rivera 2002) computes it with the **frequency-wavenumber (FK) method** (glossary
`frequency_wavenumber`). In Part 6 you compare both kinds of synthetics with the records of the ML 3.9 Sangju
earthquake that you fetched in M03.

The cell below reads the KMA catalog entry of that earthquake and the numbers of the mechanism that KIGAM
published for it (KIGAM 2021, report KEMS-2019-062; the parameters cell quotes them). M50 explained strike,
dip and rake and why a fault has two **nodal planes** (glossary `nodal_plane`); KIGAM lists both.

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {EV_LAT:.2f} N, {EV_LON:.2f} E, depth "
      f"{float(event['depth_km']):g} km, ML {float(event['ml']):.1f}")
print(f"source of the catalog: {cat.attrs.get('source', 'KMA')}")
print(f"KIGAM (2021), KEMS-2019-062: plane A strike/dip/rake {KIGAM_PLANE_A}, plane B {KIGAM_PLANE_B}, "
      f"Mw {KIGAM_MW}, depth {KIGAM_DEPTH_KM} km")

## Part 1 · A fault as a moment tensor (about 70 min, then task 1)

Slip on a small, flat fault acts on the rock like two pairs of opposite forces, a **double couple** (glossary
`double_couple`), with no net force and no net turning. Its strength is the **seismic moment** (glossary
`seismic_moment`) $M_0 = \mu A \bar{D}$: the shear modulus of the rock times the fault area times the mean slip,
in newton metres (N m). The moment tensor $\mathbf{M}$ is a symmetric 3 x 3 matrix: $M_{pq}$ is the strength of
the couple of forces that point along axis $p$ and are separated along axis $q$.

The numbers in $\mathbf{M}$ depend on the axes. Aki and Richards (2002, Box 4.4) use x = north, y = east, z =
down, the **north-east-down** frame (glossary `north_east_down`, NED). Global catalogs such as the Global CMT
project use r = up, θ = south, φ = east, the **up-south-east** frame (glossary `up_south_east`, USE), and list
$M_{rr}, M_{\theta\theta}, M_{\phi\phi}, M_{r\theta}, M_{r\phi}, M_{\theta\phi}$ (Jost and Herrmann 1989 compare the
conventions). Both describe the same forces: only the axes are turned and flipped.

For a fault of strike $\phi_s$, dip $\delta$ and rake $\lambda$, Aki and Richards (Box 4.4) give the six NED
elements in units of $M_0$:

$$\begin{aligned}
M_{xx} &= -(\sin\delta\cos\lambda\sin 2\phi_s + \sin 2\delta\sin\lambda\sin^2\phi_s), &
M_{xy} &= \sin\delta\cos\lambda\cos 2\phi_s + \tfrac12\sin 2\delta\sin\lambda\sin 2\phi_s,\\
M_{xz} &= -(\cos\delta\cos\lambda\cos\phi_s + \cos 2\delta\sin\lambda\sin\phi_s), &
M_{yy} &= \sin\delta\cos\lambda\sin 2\phi_s - \sin 2\delta\sin\lambda\cos^2\phi_s,\\
M_{yz} &= -(\cos\delta\cos\lambda\sin\phi_s - \cos 2\delta\sin\lambda\cos\phi_s), &
M_{zz} &= \sin 2\delta\sin\lambda.
\end{aligned}$$

Three properties hold for every double couple and let you test any tensor: it is symmetric, its trace (the sum of
the diagonal) is zero, and its three eigenvalues are $M_0$, $0$ and $-M_0$. The eigenvectors of $M_0$ and $-M_0$
are the T and P axes of M50 (glossary `p_t_axes`).

### M52-T1 · A moment tensor from strike, dip and rake · core · about 60 min

**Goal:** write `moment_tensor(strike_deg, dip_deg, rake_deg, m0_nm, frame="NED")`: the moment tensor of a double
couple in N m, in the north-east-down or the up-south-east frame.

**Given:** Part 1 (Aki and Richards, Box 4.4) and numpy. Not allowed: pyrocko, MTUQ, `obspy.imaging` (its
beachball and MoPaD code), instaseis, PyFK and the course package (the check compares with pyrocko).

**Contract:**
- Inputs: three angles in degrees (strike 0-360, dip 0-90, rake -180-180) and the seismic moment `m0_nm` (N m,
  positive); `frame`: `"NED"` (x north, y east, z down) or `"USE"` (r up, θ south, φ east). Any other `frame`
  raises `ValueError`.
- Output: a 3 x 3 numpy array of floats in N m, rows and columns in the order of the frame's axes (NED: north,
  east, down; USE: up, south, east).
- Examples (two invented faults): strike 30, dip 50, rake 70, `m0_nm` = 2.0e14 give in NED, in units of 1e14
  N m, [[-0.9165, 1.0634, -0.2176], [1.0634, -0.9343, -0.5025], [-0.2176, -0.5025, 1.8508]]. Strike 125, dip 35,
  rake -40, `m0_nm` = 5.0e13 give in USE, in units of 1e13 N m,
  [[-3.0201, 2.7001, 1.9396], [2.7001, 4.0910, -0.6676], [1.9396, -0.6676, -1.0708]].

**Done when** `st.check("M52-T1", func=moment_tensor)` passes on 40 made-up faults in both frames (all
angle ranges, vertical and horizontal planes, moments from 1e9 to 1e20 N m), against pyrocko's moment tensors.

**Stuck?** `st.hint("M52-T1", 1)`

In [ ]:
# Before you code: a pure thrust on a fault that dips 45 degrees (rake 90) and a pure strike-slip fault on a
# vertical plane (dip 90, rake 0): which elements of the NED tensor are zero for each, and what is the sign of
# M_zz for the thrust? Then: you turn a NED tensor into USE. Which elements keep their value, which change sign?
st.predict("M52-T1", """

""")

In [ ]:
def moment_tensor(strike_deg, dip_deg, rake_deg, m0_nm, frame="NED"):
    """Moment tensor of a double couple (N m) in the north-east-down or up-south-east frame.

    strike_deg, dip_deg, rake_deg : the fault's angles in degrees (Aki and Richards conventions)
    m0_nm                         : seismic moment (N m)
    frame                         : "NED" (north, east, down) or "USE" (up, south, east); else ValueError
    returns                       : 3 x 3 numpy array (N m), axes in the order of the frame
    """
    # Step 1: angles to radians; the six elements of Box 4.4 in units of M0 (NED), times m0_nm.
    # Step 2: for USE, write each USE axis as + or - one NED axis (up = -down, south = -north, east = east) and
    #         carry the signs into the elements (or turn the matrix with a 3 x 3 matrix of 0, 1 and -1).
    # YOUR CODE HERE
    raise NotImplementedError("M52-T1: write your code here")

In [ ]:
st.check("M52-T1", func=moment_tensor)

Your tensor for the two KIGAM planes of the Sangju earthquake (in units of $M_0$), its three tests, and the
difference between the two planes:

In [ ]:
mt_a = moment_tensor(*KIGAM_PLANE_A, 1.0)
mt_b = moment_tensor(*KIGAM_PLANE_B, 1.0)
np.set_printoptions(precision=4, suppress=True)
print("plane A, NED (units of M0):\n", mt_a)
print("symmetric:", bool(np.allclose(mt_a, mt_a.T)), "  trace:", round(float(np.trace(mt_a)), 6),
      "  eigenvalues:", np.round(np.linalg.eigvalsh(mt_a), 4))
print("plane B minus plane A, NED (units of M0):\n", mt_b - mt_a)
print("largest difference:", round(float(np.abs(mt_b - mt_a).max()), 4))
print("plane A, USE (units of M0):\n", moment_tensor(*KIGAM_PLANE_A, 1.0, frame="USE"))

**Explain (3-5 sentences):** What do the trace and the eigenvalues printed above tell you about the Sangju tensor,
and why must they come out that way for every double couple? The two KIGAM planes describe the same earthquake:
how large is the printed difference between their tensors, and where could a difference of that size come from?
Which elements are largest, and what does that say about the kind of faulting?

## Part 2 · Seismic moment and moment magnitude (about 40 min, then task 2)

The **moment magnitude** (glossary `moment_magnitude`) puts the seismic moment on a magnitude scale. Hanks and
Kanamori (1979) wrote it with $M_0$ in dyne-centimetres (glossary `dyne_centimeter`; 1 N m = $10^7$ dyne-cm). This
course uses the form of the IASPEI standard (Bormann and Dewey 2014), with $M_0$ in N m:

$$M_w = \tfrac{2}{3}\left(\log_{10} M_0 - 9.1\right).$$

Hanks and Kanamori's own constant (10.7 for dyne-cm) gives 9.05 instead of 9.1 here, a difference of 0.03 units.
One unit of $M_w$ is a factor of $10^{1.5} \approx 32$ in moment.

### M52-T2 · Moment and magnitude, both ways · core · about 30 min

**Goal:** write `magnitude_moment(mw=None, m0_nm=None)`: give exactly one of the two, get the other.

**Given:** Part 2 and numpy. Not allowed: pyrocko, MTUQ, `obspy.imaging`, instaseis, PyFK and the course package.

**Contract:**
- Inputs: either `mw` (a number or a numpy array of moment magnitudes) or `m0_nm` (seismic moments in N m,
  positive), given by keyword; giving both or neither raises `ValueError`.
- Output: with `m0_nm`, $M_w$ by the formula of Part 2; with `mw`, $M_0$ in N m (the same formula solved for
  $M_0$). A float for a single number, a numpy array of the same shape for an array. Do not round.
- Example: `m0_nm=1.0e15` gives 3.9333; `mw=5.0` gives 3.981e16 (N m).

**Done when** `st.check("M52-T2", func=magnitude_moment)` passes: 300 made-up moments from 1e6 to 1e23 N m and
magnitudes from -2 to 9.5, numbers and arrays, both directions, and the two wrong calls.

**Stuck?** `st.hint("M52-T2", 1)`

In [ ]:
# Before you code: the KIGAM report gives Mw 3.8 for the Sangju earthquake. What is its seismic moment in N m and in
# dyne-cm, roughly? By what factor does the moment change from Mw 3.8 to Mw 4.8, and from Mw 3.8 to Mw 3.9?
st.predict("M52-T2", """

""")

In [ ]:
def magnitude_moment(mw=None, m0_nm=None):
    """Moment magnitude from seismic moment (N m), or seismic moment from moment magnitude.

    mw     : moment magnitude(s), or None
    m0_nm  : seismic moment(s) in N m, or None
    returns: the other quantity (float for a number, numpy array for an array); ValueError unless exactly one
             of the two is given
    """
    # Step 1: refuse the call unless exactly one of mw and m0_nm is None.
    # Step 2: one direction is the formula of Part 2; the other is the same equation solved for M0.
    # Step 3: give back a float when the input was a single number.
    # YOUR CODE HERE
    raise NotImplementedError("M52-T2: write your code here")

In [ ]:
st.check("M52-T2", func=magnitude_moment)

Your function on the Sangju magnitude and on the test source of Parts 4-5:

In [ ]:
M0_SANGJU_NM = magnitude_moment(mw=KIGAM_MW)
M0_TEST_NM = magnitude_moment(mw=TEST_MW)
print(f"Sangju, Mw {KIGAM_MW}: M0 = {M0_SANGJU_NM:.3e} N m = {M0_SANGJU_NM * 1e7:.3e} dyne-cm")
print(f"test source, Mw {TEST_MW}: M0 = {M0_TEST_NM:.3e} N m")
print(f"Mw 3.8 -> 4.8: moment x {magnitude_moment(mw=4.8) / M0_SANGJU_NM:.1f};  3.8 -> 3.9: x "
      f"{magnitude_moment(mw=3.9) / M0_SANGJU_NM:.3f}")
print(f"Hanks and Kanamori's constant (9.05) for the Sangju moment: Mw "
      f"{(2.0 / 3.0) * (np.log10(M0_SANGJU_NM) - 9.05):.3f}")

**Explain (3-5 sentences):** Use the printed numbers: how much does the moment grow per unit of magnitude, and per
0.1 unit? A report gives Mw to one decimal: what range of moments does "Mw 3.8" cover? How large is the
difference that the constant 9.05 or 9.1 makes, compared with that rounding? What would a moment given in
dyne-cm do to your magnitude if you put it into the N m formula?

## Part 3 · The radiation pattern (about 60 min, then task 3)

A double couple does not send waves equally in all directions. Far from the source, the size and sign of each
wave depend on the direction of the ray as it leaves the source: its **takeoff angle** $i$ (glossary
`takeoff_angle`; here measured from the downward vertical, so 0 is straight down, 90 horizontal, more than 90
upward) and its **azimuth** $\phi$ (glossary `azimuth`; clockwise from north). This dependence is the **radiation
pattern** (glossary `radiation_pattern`). With the unit vector of the ray $\hat\gamma = (\sin i\cos\phi,
\sin i\sin\phi, \cos i)$ in NED, the P wave moves the ground along $\hat\gamma$ and the S wave across it. The S
motion splits into two parts (glossary `sv_sh_waves`): **SV**, along $\hat e_{SV} = (\cos i\cos\phi,
\cos i\sin\phi, -\sin i)$, the direction in which the takeoff angle grows, and **SH**, along $\hat e_{SH} =
(-\sin\phi, \cos\phi, 0)$, the direction in which the azimuth grows.

Aki and Richards (2002, eq. 4.89) give the three patterns $R^P$, $R^{SV}$, $R^{SH}$ as sums of sines and cosines of
$i$, $\phi - \phi_s$, $\delta$ and $\lambda$. They also follow from the moment tensor: with $\hat{\mathbf M} =
\mathbf M / M_0$ (NED), $R^P = \hat\gamma^{T}\hat{\mathbf M}\hat\gamma$, $R^{SV} = \hat e_{SV}^{T}\hat{\mathbf M}
\hat\gamma$ and $R^{SH} = \hat e_{SH}^{T}\hat{\mathbf M}\hat\gamma$. Either route is fine.

Two facts let you test a pattern without any formula: P is zero for every ray that lies in one of the two nodal
planes, and $|R^P|$ is largest, 1, along the T and P axes, which lie halfway (45 degrees) between the planes.

### M52-T3 · P, SV and SH radiation patterns · core · about 60 min

**Goal:** write `radiation_pattern(strike_deg, dip_deg, rake_deg, takeoff_deg, azimuth_deg)`: the far-field
radiation patterns $(R^P, R^{SV}, R^{SH})$ of a double couple.

**Given:** Part 3, numpy and your `moment_tensor` of task 1 (if you take the tensor route). Not allowed:
pyrocko, MTUQ, `obspy.imaging`, instaseis, PyFK and the course package (the check uses ObsPy's far-field
displacement vectors).

**Contract:**
- Inputs: the fault's three angles (degrees, numbers); `takeoff_deg` (0-180, from the downward vertical) and
  `azimuth_deg` (degrees clockwise from north): numbers or numpy arrays of one shape.
- Output: a tuple `(rp, rsv, rsh)` of floats for numbers, of numpy arrays of the input shape for arrays; no
  units (fractions of $M_0$); signs as in Part 3 (SV positive toward a larger takeoff angle, SH toward a larger
  azimuth).
- Example (an invented fault): strike 30, dip 50, rake 70, takeoff 120, azimuth 75 give (0.3154, 0.6230,
  -0.4208).

**Done when** `st.check("M52-T3", func=radiation_pattern)` passes on 30 made-up faults, each at 50 rays: rays in
the nodal planes (P must be zero there), the T and P axes (P must be +1 and -1), and random rays, numbers and
arrays.

**Stuck?** `st.hint("M52-T3", 1)`

In [ ]:
# Before you code: for a vertical strike-slip fault striking north (strike 0, dip 90, rake 0) and horizontal rays
# (takeoff 90), sketch R^P and R^SH against the azimuth from 0 to 360 degrees. Where are they zero and where are
# they largest? What is R^SV for these horizontal rays?
st.predict("M52-T3", """

""")

In [ ]:
def radiation_pattern(strike_deg, dip_deg, rake_deg, takeoff_deg, azimuth_deg):
    """Far-field P, SV and SH radiation patterns of a double couple (Aki and Richards conventions).

    strike_deg, dip_deg, rake_deg : the fault's angles in degrees
    takeoff_deg                   : takeoff angle(s) from the downward vertical (degrees)
    azimuth_deg                   : azimuth(s) clockwise from north (degrees)
    returns                       : (rp, rsv, rsh): floats, or numpy arrays of the input shape
    """
    # Step 1: angles to radians; work with phi - strike for the pattern formulas (or build gamma, e_SV, e_SH).
    # Step 2: either write the three sums of Aki and Richards eq. 4.89, or project the NED tensor of task 1
    #         (in units of M0) onto gamma and the two S directions.
    # Step 3: floats for numbers, arrays of the input shape for arrays.
    # YOUR CODE HERE
    raise NotImplementedError("M52-T3: write your code here")

In [ ]:
st.check("M52-T3", func=radiation_pattern)

Your patterns in map view for the KIGAM mechanism (plane A), for all downward rays seen from above (the lower
hemisphere of M50; the center is a ray straight down, the rim horizontal rays, north up), with the station
directions that PyFK gives in Part 6 marked later. The thin black line is where your $R^P$ changes sign. The rays
are placed on the net by your `equal_area` of task M50-T4: paste it into the next cell as it is in your M50
notebook (it needs numpy only), and run it.

In [ ]:
# Paste your equal_area of module M50 below this line.

In [ ]:
if not callable(globals().get("equal_area")):
    raise NameError("equal_area not found: paste your function of module M50 into the cell above and run it "
                    "(finish task M50-T4 of module M50 first).")
take_grid, az_grid = np.meshgrid(np.linspace(0.0, 90.0, 91), np.linspace(0.0, 360.0, 181))
gx, gy = equal_area(take_grid, az_grid)          # your lower-hemisphere equal-area net (M50, task 4)
patterns = radiation_pattern(*KIGAM_PLANE_A, take_grid, az_grid)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.8))
for ax, values, label in zip(axes, patterns, ("P", "SV", "SH")):
    image = ax.pcolormesh(gx, gy, values, cmap="RdBu_r", vmin=-1.0, vmax=1.0, shading="gouraud")
    ax.contour(gx, gy, patterns[0], levels=[0.0], colors="k", linewidths=0.6)
    ax.add_patch(plt.Circle((0.0, 0.0), 1.0, fill=False, color="k", lw=0.8))
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(f"{label}: largest |R| {np.abs(values).max():.2f}", fontsize=9)
fig.colorbar(image, ax=axes, shrink=0.8, label="Radiation pattern (fraction of M0)")
fig.savefig(Path(FIGURE_DIR) / "radiation_sangju.png", dpi=150)
print("largest |R^P|, |R^SV|, |R^SH| on the lower hemisphere:", [round(float(np.abs(v).max()), 3) for v in patterns])

**Explorer E12 (optional).** Move strike, dip and rake and watch your own `radiation_pattern` in map view: P, SV,
SH and the total S size, with the sign-change line of your P pattern, on your own `equal_area` net of M50. Type
station directions (azimuth, takeoff) to see where they fall. It draws only what your functions return.

In [ ]:
st.widgets.radiation_explorer(radiation=radiation_pattern, project=equal_area, mechanism=KIGAM_PLANE_A)

**Explain (3-5 sentences):** In the maps above, where do the black sign-change lines of P run, and what are they
on the focal sphere? Where are P and S largest compared with each other? In the explorer, turn the mechanism into
KIGAM's plane B: what changes in the maps, and why? Which of the two planes is the fault cannot be told from the
far-field pattern: say why in one sentence.

## Part 4 · A far-field synthetic in a whole space (about 80 min, then task 4)

In a uniform, unbounded rock (density $\rho$, P speed $\alpha$, S speed $\beta$), a double couple whose moment
grows as $M_0(t)$ makes, at distance $r$, the displacement (Aki and Richards 2002, eq. 4.29, far-field terms)

$$\mathbf u(\mathbf x, t) = \frac{R^P\,\hat\gamma}{4\pi\rho\alpha^3 r}\,\dot M_0\!\left(t - \frac{r}{\alpha}\right) +
\frac{R^{SV}\hat e_{SV} + R^{SH}\hat e_{SH}}{4\pi\rho\beta^3 r}\,\dot M_0\!\left(t - \frac{r}{\beta}\right).$$

Read it piece by piece. The **far field** (glossary `far_field`) decays as $1/r$ (**geometric spreading**,
glossary `geometric_spreading`). Each wave arrives after its travel time $r/\alpha$ or $r/\beta$. Its shape is
the moment **rate** $\dot M_0(t) = M_0\, s(t)$, where $s(t)$ is the source time function scaled to an area of 1:
a far-field displacement pulse has the shape of the moment rate, not of the moment. The size of each wave is its
radiation pattern divided by the cube of its speed, so S is much larger than P.

The full eq. 4.29 has two more terms, which decay as $1/r^2$ and $1/r^4$: the **near field** (glossary
`near_field`). You leave them out here and measure in Part 5 what that costs.

The test source of Parts 4-5 is the made-up fault `TEST_MECHANISM` with `TEST_MW`, whose moment rate is a smooth
Hann pulse (a raised cosine) lasting `WS_STF_S` seconds. The next cell builds that pulse: `stf_rate` holds $s(t)$
at the times `stf_t` (starting at the origin), with area 1.

In [ ]:
stf_t = np.arange(int(round(WS_STF_S / WS_DT_S)) + 1) * WS_DT_S
stf_rate = np.hanning(stf_t.size)
stf_rate = stf_rate / (stf_rate.sum() * WS_DT_S)
print(f"moment-rate pulse: {stf_t.size} samples at {WS_DT_S} s, area {stf_rate.sum() * WS_DT_S:.6f}, "
      f"peak {stf_rate.max():.4f} per second at {stf_t[np.argmax(stf_rate)]:.2f} s")

### M52-T4 · Far-field displacement of a point source · core · about 90 min

**Goal:** write `farfield_displacement(t_s, offset_km, mechanism, m0_nm, stf_rate, stf_dt_s, vp_km_s, vs_km_s,
rho_kg_m3)`: the far-field P plus S displacement of Part 4 at one receiver, in metres, in north-east-down.

**Given:** Part 4, numpy and your `radiation_pattern` of task 3. Not allowed: pyrocko, MTUQ, `obspy.imaging`,
PyFK, instaseis and the course package (the check builds its own truth from ObsPy's far-field vectors and a
moment tensor of pyrocko).

**Contract:**
- Inputs: `t_s`, a 1-D numpy array of times after the origin (s); `offset_km`, (north, east, down) of the
  receiver minus the source (km, not all zero); `mechanism`, (strike, dip, rake) in degrees; `m0_nm` (N m);
  `stf_rate`, the samples of $s(t)$ (area 1, in 1/s) at the times 0, `stf_dt_s`, 2 `stf_dt_s`, ... after the
  origin; speeds in km/s and density in kg/m³.
- Use SI units inside (r in m, speeds in m/s). Take the takeoff angle and the azimuth of the straight ray from the
  offset (from the downward vertical; clockwise from north), and the delayed $s(t - \tau)$ by linear interpolation
  between its samples (`numpy.interp`), zero before 0 and after the last sample.
- Output: a numpy array of shape (3, `len(t_s)`): north, east and down displacement in m. Do not change the
  inputs.
- Example (invented): strike 0, dip 90, rake 0, `m0_nm` = 1e15, a receiver 10 km to the north-east at the source
  depth (offset (7.0711, 7.0711, 0)), 6.0 and 3.5 km/s, 2700 kg/m³, and a 1-s triangle for $s$ (peak 2 per second
  at 0.5 s): the P pulse peaks at 2.729e-5 m along (0.7071, 0.7071, 0) at 2.167 s, and the S motion is zero there.

**Done when** `st.check("M52-T4", func=farfield_displacement)` passes on 20 made-up receivers (above, below and
beside the source, 1-200 km), rocks and pulses, against the far-field terms of eq. 4.29 built from pyrocko's
moment tensor and ObsPy's far-field vectors.

**Stuck?** `st.hint("M52-T4", 1)`

In [ ]:
# Before you code: at a receiver in the direction where |R^P| = 1, how much larger is the S pulse than the P pulse
# would be in a direction where |R^SH| = 1, with 6.0 and 3.5 km/s? How does the peak displacement change from 10 km
# to 20 km? If the pulse lasts twice as long with the same moment, what happens to the peak displacement?
st.predict("M52-T4", """

""")

In [ ]:
def farfield_displacement(t_s, offset_km, mechanism, m0_nm, stf_rate, stf_dt_s, vp_km_s, vs_km_s, rho_kg_m3):
    """Far-field P + S displacement (m, north-east-down) of a double couple in a whole space.

    t_s       : 1-D array of times after the origin (s)
    offset_km : (north, east, down) of the receiver minus the source (km)
    mechanism : (strike, dip, rake) in degrees
    m0_nm     : seismic moment (N m)
    stf_rate  : s(t) at 0, stf_dt_s, 2 stf_dt_s, ... (area 1, 1/s)
    vp_km_s, vs_km_s, rho_kg_m3 : the rock
    returns   : numpy array (3, len(t_s)): north, east, down displacement (m)
    """
    # Step 1: r (m), the unit vector gamma, the takeoff angle and the azimuth of the ray.
    # Step 2: R^P, R^SV, R^SH from task 3; the unit vectors e_SV and e_SH of Part 3.
    # Step 3: s(t - r/alpha) and s(t - r/beta) by numpy.interp on the pulse's own times.
    # Step 4: the two terms of Part 4 with m0_nm, rho and the speeds in SI units; add them.
    # YOUR CODE HERE
    raise NotImplementedError("M52-T4: write your code here")

In [ ]:
st.check("M52-T4", func=farfield_displacement)

Your function for the test source at the four receivers of Part 5 (the rock `WS_*`, receivers `WS_RECEIVER_KM`
deep, the source `WS_SOURCE_KM` deep). The cell turns north-east-down into the three directions that PyFK uses:
Z up, R (radial) away from the source, T (transverse) 90 degrees clockwise from R seen from above (glossary
`radial_transverse`).

In [ ]:
def ned_to_zrt(u_ned, azimuth_deg):
    """(north, east, down) rows -> (Z up, R away from the source, T clockwise from R) rows."""
    a = np.radians(azimuth_deg)
    north, east, down = u_ned
    return np.array([-down, north * np.cos(a) + east * np.sin(a), -north * np.sin(a) + east * np.cos(a)])


ws_t = np.arange(0.0, 40.0, WS_DT_S)
fig, axes = plt.subplots(len(WS_DISTANCES_KM), 1, figsize=(7.5, 7.0), sharex=True)
for ax, x_km, az in zip(axes, WS_DISTANCES_KM, WS_AZIMUTHS_DEG):
    offset = (x_km * np.cos(np.radians(az)), x_km * np.sin(np.radians(az)), WS_RECEIVER_KM - WS_SOURCE_KM)
    zrt = ned_to_zrt(farfield_displacement(ws_t, offset, TEST_MECHANISM, M0_TEST_NM, stf_rate, WS_DT_S,
                                           WS_VP, WS_VS, WS_RHO), az)
    r_km = float(np.linalg.norm(offset))
    for row, name, color in zip(zrt, "ZRT", ("k", "tab:blue", "tab:orange")):
        ax.plot(ws_t, row * 1e6, color=color, lw=0.8, label=name)
    ax.set_ylabel("Disp. (µm)")
    ax.set_title(f"x = {x_km:g} km, r = {r_km:.1f} km", fontsize=8, loc="left")
    print(f"x {x_km:5.1f} km, r {r_km:6.1f} km: largest |u| {np.abs(zrt).max():.3e} m; "
          f"P at {r_km / WS_VP:.2f} s, S at {r_km / WS_VS:.2f} s")
axes[-1].set_xlabel("Time after origin (s)")
axes[0].legend(ncol=3, fontsize=7, loc="upper right")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "farfield_test_source.png", dpi=150)

**Explain (3-5 sentences):** From the printed peaks, how does the largest displacement change with r between the
receivers, and is that what 1/r predicts once the radiation pattern is taken into account? Why is the S pulse so
much larger than the P pulse? What shape does each pulse have, and why that shape rather than a step?

## Part 5 · PyFK in its own environment (about 90 min, then task 5)

PyFK needs Python 3.9, so it lives in a separate **worker environment** (glossary `worker_environment`):
`bin/sgtl-install pyfk --apply` made it, and `sgtl_tutorial.pyfk_runner` (imported as `fk` in the setup cell) runs
it for you. `fk.run(...)` writes your request as JSON, starts PyFK's own Python on it with one core and low
priority, and reads the result back from plain files (`result.npz`, `result.json`). It hands the source list to
PyFK unchanged and gives back exactly what PyFK's `calculate_sync` returns: arrays `z`, `r`, `t` (one row per
receiver), the time of each row's first sample after the origin, and PyFK's first P and S arrival times. Read
`help(fk.run)` once.

PyFK's model always has a **free surface** on top (glossary `free_surface`): the ground surface, where waves
reflect. To compare with a whole space, the next cell puts one uniform layer over nothing (a half-space), buries
the source `WS_SOURCE_KM` deep and the receivers `WS_RECEIVER_KM` deep, and keeps only the time before the first
echo from the surface reaches a receiver. The layer rows are PyFK's: thickness (0 = goes on forever), S speed,
P speed, density (g/cm³) and the **quality factors** Qs and Qp (glossary `quality_factor`; a very large Q means
no loss of energy, as in the formula of Part 4). PyFK picks its **wavenumber step** (glossary `wavenumber_step`)
from the largest distance of a call, so all receivers here are computed in one call.

In [ ]:
WS_LAYERS = [[0.0, WS_VS, WS_VP, WS_RHO / 1000.0, 1.0e5, 1.0e5]]
ws_pulse = list(stf_rate * WS_DT_S)          # PyFK adds its pulse samples without the time step: they sum to 1
ws_echo_s = [float(np.hypot(x, WS_SOURCE_KM + WS_RECEIVER_KM)) / WS_VP for x in WS_DISTANCES_KM]
print("PyFK's Python:", short_path(fk.pyfk_python()))
print("layer row (thickness, Vs, Vp, density, Qs, Qp):", WS_LAYERS[0])
print("first surface echo at each receiver (s):", [round(v, 2) for v in ws_echo_s])


def ws_run(source, out_dir):
    """PyFK for the whole-space test: the four receivers of the parameters cell, one call."""
    return fk.run(WS_LAYERS, source_depth_km=WS_SOURCE_KM, receiver_depth_km=WS_RECEIVER_KM,
                  distances_km=WS_DISTANCES_KM, azimuths_deg=WS_AZIMUTHS_DEG, source=source, dt_s=WS_DT_S,
                  npt=WS_NPT, stf=ws_pulse, out_dir=out_dir)

### M52-T5 · Displacement from PyFK, for a moment in N m · core · about 90 min

**Goal:** write `pyfk_displacement(m0_nm, strike_deg, dip_deg, rake_deg, run)`: the ground displacement in metres
that PyFK predicts for a double couple of seismic moment `m0_nm` (N m), on PyFK's own samples.

**Given:** `run(source)`, which hands the list `source` to PyFK as its double-couple source (four numbers:
something, strike, dip, rake) and returns `(z, r, t, dt_s)`: `calculate_sync`'s samples unchanged (1-D for one
receiver, 2-D with one row per receiver) and their sample interval. Your `magnitude_moment` of task 2. PyFK's own
code: the cell after the trap prints where it is; read `SourceModel.calculate_radiation_pattern` (in
`pyfk/config/config.py`) and `calculate_sync` (in `pyfk/sync/sync.py`). Your `farfield_displacement` of task 4, for
a measurement of your own. A colleague's cell below uses PyFK for the test source: run it as it is before your
prediction. Not allowed: the course package (use `run`, do not start PyFK yourself), pyrocko and MTUQ.

**Contract:**
- Inputs: `m0_nm` > 0; the fault's angles in degrees; `run` as above.
- Call `run` once, with a four-number source whose first number is what PyFK expects there for this moment.
- Output: a tuple `(uz, ur, ut)` of numpy arrays with the shape of `run`'s arrays: ground displacement in m along
  Z (up), R and T, on the same samples, measured from where the ground was before the first arrival (`run`'s
  first sample comes before it).
- Find out from PyFK's code and from a measurement of your own (not from a docstring) what the first number must
  be and what kind of motion, in which unit, `calculate_sync` returns.

**Done when** `st.check("M52-T5", func=pyfk_displacement)` passes on 12 made-up cases: the check's `run` behaves
exactly like PyFK in what the first number means and in the kind and unit of motion it returns (made-up
Green's functions, so no PyFK is started), with one and with several receivers.

**Stuck?** `st.hint("M52-T5", 1)`

In [ ]:
# A colleague's cell: PyFK synthetics of the test source at the four receivers. Run it as it is.
colleague = ws_run([M0_TEST_NM, *TEST_MECHANISM], Path(RESULT_DIR) / "fk_colleague")
colleague_disp_cm = colleague.z          # "calculate_sync returns displacement in cm, its docstring says so"
peak_mm = np.abs(colleague_disp_cm).max(axis=1) * 10.0
print("colleague: peak vertical displacement (mm) at", WS_DISTANCES_KM, "km:", peak_mm)
too_big = peak_mm > 10.0
print("colleague's sanity check (no receiver above 10 mm):", "passed" if not too_big.any() else "FAILED")

In [ ]:
pyfk_code = fk.pyfk_python().parent.parent / "lib" / "python3.9" / "site-packages" / "pyfk"
print("PyFK's code:", short_path(pyfk_code))
print("  source models:", "config/config.py (class SourceModel, method calculate_radiation_pattern)")
print("  synthetics:   ", "sync/sync.py (functions calculate_sync and sync_calculate_gf)")

In [ ]:
# Before you code: what did the colleague's cell print, and why did the sanity check say what it said? Once the
# first number of the source is right, will the colleague's "displacement in cm" match your far-field displacement
# of task 4 at the 80-km receiver? If not, what would you have to do to PyFK's output to compare it with yours?
st.predict("M52-T5", """

""")

In [ ]:
def pyfk_displacement(m0_nm, strike_deg, dip_deg, rake_deg, run):
    """Ground displacement (m) from PyFK for a double couple of seismic moment m0_nm (N m).

    m0_nm      : seismic moment (N m)
    strike_deg, dip_deg, rake_deg : the fault (degrees)
    run        : run(source) -> (z, r, t, dt_s), PyFK's calculate_sync samples for that source, unchanged
    returns    : (uz, ur, ut): displacement in m on run's samples (same shape)
    """
    # Step 1: the four-number source: what PyFK's calculate_radiation_pattern expects first, then the angles.
    # Step 2: one call of run.
    # Step 3: turn each array into displacement in m (your measurement decides how).
    # YOUR CODE HERE
    raise NotImplementedError("M52-T5: write your code here")

In [ ]:
st.check("M52-T5", func=pyfk_displacement)

Now the measurement with the real PyFK: your `pyfk_displacement` for the test source, against your far-field
displacement of task 4, at the four receivers, from PyFK's first sample to 0.5 s before the first surface echo.
For each receiver the cell prints the distance r and r divided by the S wavelength (S speed times the pulse
duration), the correlation coefficient of each component (glossary `correlation_coefficient`), the ratio of the
largest PyFK displacement to your largest (all three components), the displacement left at the end of the window
as a fraction of the peak, and the relative misfit of the three components together, in displacement and in
velocity (your displacement differentiated with `numpy.gradient`, PyFK's samples turned into m/s).

In [ ]:
holder = {}


def ws_pyfk(source):
    """run(source) for pyfk_displacement: the whole-space test receivers, as (z, r, t, dt_s)."""
    res = ws_run(source, Path(RESULT_DIR) / "fk_whole_space")
    holder["res"] = res
    return res.z, res.r, res.t, res.dt_s


fk_disp = np.array(pyfk_displacement(M0_TEST_NM, *TEST_MECHANISM, ws_pyfk))     # (3, receivers, samples)
ws_res = holder["res"]
rows = []
fig, axes = plt.subplots(len(WS_DISTANCES_KM), 3, figsize=(10.0, 7.5), sharex="row")
for k, (x_km, az) in enumerate(zip(WS_DISTANCES_KM, WS_AZIMUTHS_DEG)):
    times = ws_res.times(k)
    offset = (x_km * np.cos(np.radians(az)), x_km * np.sin(np.radians(az)), WS_RECEIVER_KM - WS_SOURCE_KM)
    mine = ned_to_zrt(farfield_displacement(times, offset, TEST_MECHANISM, M0_TEST_NM, stf_rate, WS_DT_S,
                                            WS_VP, WS_VS, WS_RHO), az)
    theirs = fk_disp[:, k, :]
    keep = times < ws_echo_s[k] - 0.5
    mine_v, theirs_v = np.gradient(mine, WS_DT_S, axis=1), np.array([ws_res.z[k], ws_res.r[k], ws_res.t[k]]) / 100.0
    r_km = float(np.linalg.norm(offset))
    row = {"x_km": x_km, "r_km": round(r_km, 1), "r_over_wavelength": round(r_km / (WS_VS * WS_STF_S), 1)}
    for c, name in enumerate("ZRT"):
        row[f"cc_{name}"] = round(float(np.corrcoef(mine[c, keep], theirs[c, keep])[0, 1]), 3)
    row["peak_ratio"] = round(float(np.abs(theirs[:, keep]).max() / np.abs(mine[:, keep]).max()), 3)
    end = int(np.flatnonzero(keep)[-1])
    row["end_over_peak"] = round(float(np.abs(theirs[:, end]).max() / np.abs(theirs[:, keep]).max()), 3)
    row["misfit_disp"] = round(float(np.linalg.norm((theirs - mine)[:, keep]) / np.linalg.norm(theirs[:, keep])), 3)
    row["misfit_vel"] = round(float(np.linalg.norm((theirs_v - mine_v)[:, keep]) / np.linalg.norm(theirs_v[:, keep])), 3)
    rows.append(row)
    for c, name in enumerate("ZRT"):
        ax = axes[k, c]
        ax.plot(times[keep], theirs[c, keep] * 1e6, color="k", lw=0.9, label="PyFK")
        ax.plot(times[keep], mine[c, keep] * 1e6, color="tab:red", lw=0.8, ls="--", label="far field")
        ax.set_title(f"{name}, r = {r_km:.0f} km", fontsize=8, loc="left")
    axes[k, 0].set_ylabel("Disp. (µm)")
for ax in axes[-1]:
    ax.set_xlabel("Time after origin (s)")
axes[0, 0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "pyfk_vs_farfield.png", dpi=150)
ws_table = pd.DataFrame(rows)
print(ws_table.to_string(index=False))

**Explain (3-5 sentences):** What did the colleague's cell print, why, and why did its sanity check not catch it?
What did your own measurement show about the kind and unit of motion that PyFK returns? In the table, how do the
misfits and the displacement left at the end of the window change with r over the wavelength, and which terms of
eq. 4.29 cause that? Give the tolerance (correlation and amplitude ratio) within which you would accept your
far-field synthetic as a replacement for PyFK, and from which r on, using the printed numbers.

## Part 6 · Sangju: layered-Earth synthetics against the records (about 120 min, then task 6)

Now the real earthquake. PyFK computes the synthetics of the KIGAM mechanism (plane A, `KIGAM_MW`, source
`KIGAM_DEPTH_KM` deep) in the `kim1983` crust at the stations `FIT_STATIONS`, at their distances and azimuths from
the KMA epicenter. `fk.model_layers` turns the course's model into PyFK's rows. The model lists speeds only, so
each layer's density comes from its P speed by a straight-line rule of thumb, density = 0.77 + 0.32 Vp (g/cm³,
with Vp in km/s; Berteussen 1977): a layer with Vp 6.0 km/s gets 2.69 g/cm³. It is the same rule that the test run
of `bin/sgtl-install pyfk --apply` used when you installed PyFK. The quality factors are Qs 500 and Qp 1000. The
source time function is PyFK's trapezoid of `STF_DURATION_S` seconds.

Synthetics and records must be made comparable. The records are in counts: the cell after the PyFK run removes the
instrument response as in M10 (glossary `response_removal`; StationMetadata by Byeong-Seok Ahn, PKNU, Ahn et al.
2024), to ground velocity in m/s, with the `pre_filter` `PRE_FILT_HZ`. In Part 5 you found what PyFK's samples are;
the comparison of this part is in ground velocity, so PyFK's samples need only their unit changed.

In [ ]:
station_rows = []
for code in FIT_STATIONS:
    path = sorted((st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN.strftime("%Y%m%d%H%M%S") / BAND)
                  .glob(f"*.{code}.{BAND}Z.sac"))
    if not path:
        raise FileNotFoundError(f"No {BAND}Z file of {code} in your {DATASET} download: fetch the data with STP "
                                "(module M03), or change FIT_STATIONS in the parameters cell.")
    sac = read(str(path[0]), headonly=True)[0].stats.sac
    dist_m, az, baz = gps2dist_azimuth(EV_LAT, EV_LON, float(sac.stla), float(sac.stlo))
    station_rows.append({"station": code, "distance_km": round(dist_m / 1000.0, 3), "azimuth_deg": round(az, 2),
                         "back_azimuth_deg": round(baz, 2)})
stations = pd.DataFrame(station_rows)
LAYERS = fk.model_layers(st.data.velocity_model(VELOCITY_MODEL))
FK_PULSE = fk.trapezoid_stf(STF_DURATION_S, 0.5, FK_DT_S)
print(stations.to_string(index=False))
print(f"{VELOCITY_MODEL} rows (thickness km, Vs, Vp, density, Qs, Qp):")
for layer in LAYERS:
    print("  ", [round(v, 3) for v in layer])


def sangju_run(layers, source, out_dir):
    """PyFK for the Sangju stations: one call with every distance."""
    return fk.run(layers, source_depth_km=KIGAM_DEPTH_KM, distances_km=stations["distance_km"].tolist(),
                  azimuths_deg=stations["azimuth_deg"].tolist(), source=source, dt_s=FK_DT_S, npt=FK_NPT,
                  stf=FK_PULSE, out_dir=out_dir)


FK_MAIN = Path(RESULT_DIR) / "fk_sangju_kim1983"
sangju = sangju_run(LAYERS, [KIGAM_MW, *KIGAM_PLANE_A], FK_MAIN)
pattern_here = radiation_pattern(*KIGAM_PLANE_A, sangju.takeoff_p_deg, stations["azimuth_deg"].to_numpy())
print(pd.DataFrame({"station": stations["station"], "takeoff_P_deg": np.round(sangju.takeoff_p_deg, 1),
                    "R_P": np.round(pattern_here[0], 3), "R_SV": np.round(pattern_here[1], 3),
                    "R_SH": np.round(pattern_here[2], 3)}).to_string(index=False))

The records, as ground velocity: demean, 5 % Hann taper and response removal to velocity with `PRE_FILT_HZ` and
`water_level=None`, as in M10 (some channels print the evalresp warning about sensitivities that M03 explained;
the removal keeps the sign of the counts).

In [ ]:
SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN.strftime("%Y%m%d%H%M%S") / BAND
XML_DIR = st.data.root() / "metadata" / "stationxml"


def velocity_stream(code):
    """Ground velocity (m/s) of the three components of one station (M10's recipe)."""
    stream = read(str(SAC_DIR / f"*.{code}.{BAND}?.sac"))
    stream.detrend("demean").taper(max_percentage=0.05, type="hann")
    stream.remove_response(inventory=read_inventory(str(XML_DIR / f"{code}.xml")), output="VEL",
                           pre_filt=PRE_FILT_HZ, water_level=None)
    return stream


records = {code: velocity_stream(code) for code in FIT_STATIONS}
for code, stream in records.items():
    print(code, [f"{tr.stats.channel}: {np.abs(tr.data).max():.3e} m/s" for tr in stream])

To compare shapes you need the time shift between record and synthetic, which you measured in M41 with
cross-correlation (glossary `cross_correlation`). Paste your `xcorr_lag` of task M41-T2 into the next cell as it
is in your M41 notebook (it needs numpy only), and run it.

In [ ]:
# Paste your xcorr_lag of module M41 below this line.

In [ ]:
if not callable(globals().get("xcorr_lag")):
    raise NameError("xcorr_lag not found: paste your function of module M41 into the cell above and run it "
                    "(finish task M41-T2 of module M41 first).")
print("xcorr_lag is ready:", xcorr_lag.__doc__.strip().splitlines()[0] if xcorr_lag.__doc__ else "(no docstring)")

### M52-T6 · A synthetic against a record · core · about 120 min

**Goal:** write `station_fit(obs, syn, origin, back_azimuth_deg, band_hz, window_s, max_lag_s, xcorr)`: for one
station, the correlation, the time shift and the amplitude ratio of record and synthetic on Z, R and T.

**Given:** Parts 5 and 6, ObsPy, numpy, pandas, and your `xcorr_lag` of M41 (passed in as `xcorr`). Not allowed:
the course package.

**Contract:**
- Inputs: `obs`, an ObsPy Stream with the components Z, N and E of one station, ground velocity in m/s (do not
  change it); `syn`, a dict with `"z"`, `"r"`, `"t"` (1-D arrays of PyFK's samples for this station, unchanged,
  in the unit you found in Part 5), `"b_s"` (seconds after the origin of their first sample) and `"dt_s"`;
  `origin`, the origin time (UTCDateTime); `back_azimuth_deg`, from the station toward the epicenter (degrees
  clockwise from north); `band_hz`, (low, high) in Hz; `window_s`, (start, end) in s after the origin, inside the
  synthetic's span; `max_lag_s`; `xcorr`, your `xcorr_lag` of module M41 (same contract).
- Steps: turn the record's N and E into R and T with ObsPy's `rotate("NE->RT", back_azimuth=...)` (R away from
  the source, T 90 degrees clockwise from R, as PyFK's); the synthetic into m/s; band-pass each whole record at
  its own sampling (ObsPy `filter("bandpass", ..., corners=4, zerophase=True)`); then put the filtered record on
  the synthetic's sample times (`numpy.interp`, both as seconds after `origin`), and keep the samples whose time
  lies in `window_s` (ends included).
- For each component: `lag_s, cc = xcorr(syn_window, obs_window, dt_s, max_lag_s)[:2]` (so a positive lag means
  the record is later than the synthetic), and `amp_ratio` = largest |record| / largest |synthetic| in the window.
- Output: a pandas DataFrame with the index `["Z", "R", "T"]` and the columns `cc`, `lag_s`, `amp_ratio`.
- Example (invented): a record that is the synthetic delayed by 0.30 s and multiplied by 0.5 (in m/s) gives
  cc 1.0, lag_s 0.30 and amp_ratio 0.5 on each component.

**Done when** `st.check("M52-T6", func=station_fit, xcorr=xcorr_lag, fit_json=FIT_JSON)` passes: 10 made-up
stations (records built from made-up synthetics with known shifts, factors and back azimuths, at 100 samples per
second), then your tables of the Sangju stations, which the check recomputes with your function and your
`xcorr_lag` from your PyFK runs and your records, for `VELOCITY_MODEL` and for `CHANGED_MODEL`; it also recomputes
the SHA-256 fingerprints of the three runs of R3 (the first run, its unchanged rerun and the changed-model run)
from their files (it prints no value of its own).

**Stuck?** `st.hint("M52-T6", 1)`

In [ ]:
# Before you code: look at the R_P, R_SV and R_SH printed for the six stations. At which station and on which
# components do you expect the synthetic to be very small, and what will that do to the amplitude ratio and the
# correlation there? On the other stations, do you expect the record to be larger or smaller than the synthetic, by
# roughly what factor, and would you expect the record to be early or late compared with a synthetic computed for
# the KMA origin time?
st.predict("M52-T6", """

""")

In [ ]:
def station_fit(obs, syn, origin, back_azimuth_deg, band_hz, window_s, max_lag_s, xcorr):
    """Correlation, lag and amplitude ratio of a record against a PyFK synthetic on Z, R and T.

    obs              : ObsPy Stream of one station (Z, N, E), ground velocity in m/s; unchanged
    syn              : dict with "z", "r", "t" (PyFK's samples), "b_s" and "dt_s"
    origin           : UTCDateTime of the origin
    back_azimuth_deg : from the station toward the epicenter (degrees)
    band_hz          : (low, high) band-pass corners (Hz)
    window_s         : (start, end) in s after the origin
    max_lag_s        : largest lag searched (s)
    xcorr            : your xcorr_lag of module M41 (same contract)
    returns          : DataFrame, index Z, R, T; columns cc, lag_s, amp_ratio
    """
    # Step 1: a copy of the record turned to Z, R, T; band-pass it at its own sampling.
    # Step 2: the synthetic in m/s as ObsPy Traces (delta = dt_s), band-passed the same way.
    # Step 3: the record's times after the origin; numpy.interp onto the synthetic's times; the window.
    # Step 4: for each component: xcorr(synthetic, record, ...) and the ratio of the largest absolute values.
    # YOUR CODE HERE
    raise NotImplementedError("M52-T6: write your code here")

Your function on every station: the window runs from 2 s before PyFK's first P arrival to 10 s after its first
S arrival.

**R3: the production run again, unchanged, then with one parameter changed.** The same cell also makes the two
runs of the last rung. First the same request into a new folder: a program that computes the same thing twice
should give the same numbers. The **checksum** (glossary `checksum`, SHA-256) of a result file is a short
fingerprint of every byte in it, so two equal fingerprints show two identical files. Then the one change: the
velocity model `CHANGED_MODEL` instead of `VELOCITY_MODEL`, everything else the same, and your function on every
station of that run too.

The cell writes your two tables, the three run folders and the fingerprints of their `result.npz` to `FIT_JSON`,
which the check reads. It prints only what it wrote; the numbers come after the check.

In [ ]:
FIT_JSON = Path(RESULT_DIR) / "M52_sangju_fit.json"
FK_AGAIN = Path(RESULT_DIR) / "fk_sangju_kim1983_again"
FK_CHANGED = Path(RESULT_DIR) / f"fk_sangju_{CHANGED_MODEL}"


def sha256(path):
    """SHA-256 fingerprint of a file, as 64 hexadecimal digits."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def fit_table(result, label):
    """station_fit on every station for one PyFK result; one row per station and component."""
    out = []
    for k, row in stations.iterrows():
        syn = {"z": result.z[k], "r": result.r[k], "t": result.t[k], "b_s": float(result.b_s[k]),
               "dt_s": result.dt_s}
        window = (float(result.t_p_s[k]) - 2.0, float(result.t_s_s[k]) + 10.0)
        table = station_fit(records[row["station"]], syn, ORIGIN, row["back_azimuth_deg"], FIT_BAND_HZ, window,
                            MAX_LAG_S, xcorr_lag)
        for comp, values in table.iterrows():
            out.append({"run": label, "station": row["station"], "component": comp,
                        "window_s": list(window), **values.to_dict()})
    return out


fit_rows = fit_table(sangju, VELOCITY_MODEL)
again = sangju_run(LAYERS, [KIGAM_MW, *KIGAM_PLANE_A], FK_AGAIN)
changed_layers = fk.model_layers(st.data.velocity_model(CHANGED_MODEL))
changed = sangju_run(changed_layers, [KIGAM_MW, *KIGAM_PLANE_A], FK_CHANGED)
changed_rows = fit_table(changed, CHANGED_MODEL)
FIT_JSON.write_text(json.dumps({
    "dataset": DATASET, "band": BAND, "event_folder": ORIGIN.strftime("%Y%m%d%H%M%S"), "origin": str(ORIGIN),
    "epicenter": [EV_LAT, EV_LON], "pre_filt_hz": list(PRE_FILT_HZ), "taper": 0.05, "band_hz": list(FIT_BAND_HZ),
    "max_lag_s": MAX_LAG_S, "fk_dir": str(FK_MAIN), "fk_sha256": sha256(FK_MAIN / "result.npz"),
    "stations": stations.to_dict(orient="records"), "rows": fit_rows,
    "rerun": {"fk_dir": str(FK_AGAIN), "fk_sha256": sha256(FK_AGAIN / "result.npz")},
    "changed": {"model": CHANGED_MODEL, "fk_dir": str(FK_CHANGED), "fk_sha256": sha256(FK_CHANGED / "result.npz"),
                "rows": changed_rows}}, indent=1), encoding="utf-8")
print(f"wrote {FIT_JSON}: {len(fit_rows)} rows for {VELOCITY_MODEL}, {len(changed_rows)} rows for "
      f"{CHANGED_MODEL}, fingerprints of 3 PyFK runs")

In [ ]:
st.check("M52-T6", func=station_fit, xcorr=xcorr_lag, fit_json=FIT_JSON)

Your table and a figure of records (black) and synthetics (red), band-passed `FIT_BAND_HZ`, each pair scaled by
the record's largest value in the window, the synthetic shifted by your lag:

In [ ]:
fit = pd.DataFrame(fit_rows)
print(fit.drop(columns=["run", "window_s"]).round(3).to_string(index=False))
print("median cc per component:", fit.groupby("component")["cc"].median().round(3).to_dict())
print("median amplitude ratio per component:", fit.groupby("component")["amp_ratio"].median().round(3).to_dict())

fig, axes = plt.subplots(len(FIT_STATIONS), 3, figsize=(10.0, 1.5 * len(FIT_STATIONS) + 1.0), sharex="row")
for k, code in enumerate(FIT_STATIONS):
    rec = records[code].copy().rotate("NE->RT", back_azimuth=float(stations.loc[k, "back_azimuth_deg"]))
    rec.filter("bandpass", freqmin=FIT_BAND_HZ[0], freqmax=FIT_BAND_HZ[1], corners=4, zerophase=True)
    times = sangju.times(k)
    window = (float(sangju.t_p_s[k]) - 2.0, float(sangju.t_s_s[k]) + 10.0)
    for c, (comp, key) in enumerate((("Z", "z"), ("R", "r"), ("T", "t"))):
        model = Trace(data=sangju.component(key)[k] / 100.0)
        model.stats.delta = sangju.dt_s
        model.filter("bandpass", freqmin=FIT_BAND_HZ[0], freqmax=FIT_BAND_HZ[1], corners=4, zerophase=True)
        trace = rec.select(component=comp)[0]
        keep = (times >= window[0]) & (times <= window[1])
        on_grid = np.interp(times, trace.times(reftime=ORIGIN), trace.data)
        scale = np.abs(on_grid[keep]).max()
        this = fit[(fit["station"] == code) & (fit["component"] == comp)].iloc[0]
        ax = axes[k, c]
        ax.plot(times[keep], on_grid[keep] / scale, color="k", lw=0.8)
        ax.plot(times[keep] + this["lag_s"], model.data[keep] / scale, color="tab:red", lw=0.8)
        ax.set_title(f"{code} {comp}: cc {this['cc']:.2f}, ratio {this['amp_ratio']:.2f}", fontsize=7, loc="left")
        ax.set_yticks([])
for ax in axes[-1]:
    ax.set_xlabel("Time after origin (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "sangju_fit.png", dpi=150)

**The amplitude ratio as a magnitude.** PyFK's synthetics grow in proportion to the seismic moment it is given
(the source time function keeps its shape), so a ratio $q$ = record / synthetic is the ratio that a moment $q$
times the one used would give. On the magnitude scale of Part 2 that is a change $\Delta M_w = \tfrac23
\log_{10} q$: $q$ = 0.5 would mean a magnitude 0.20 smaller. The next cell prints the moment used, where it comes
from, and the $\Delta M_w$ of the median ratio per component and over all rows. It also prints the factor that the
last digit of a magnitude given to 0.1 allows ($\pm 0.05$ in $M_w$).

How to read it: one smaller moment scales every station and every component by the same factor, so it can
explain your ratios only if the three components give about the same $\Delta M_w$ and the size of that
$\Delta M_w$ is one the published magnitude allows. Other causes change the ratio too and the notebook does not
separate them: the response removal (a wrong gain changes one station, all its components), the band (if the
true source lasted longer than the trapezoid of `STF_DURATION_S` s, the records would be smaller in
`FIT_BAND_HZ` than the synthetics without any change of moment), the damping of the crust (the quality factors
used are large, which means little loss) and the ground under each station (soft rock near the surface makes
records larger, not smaller). Say which of these your printed numbers can and cannot tell apart.

In [ ]:
m0_used = magnitude_moment(mw=KIGAM_MW)
print(f"synthetics: Mw {KIGAM_MW} (KIGAM 2021, report KEMS-2019-062), M0 = {m0_used:.3e} N m "
      "(your magnitude_moment of task 2)")
median_q = fit.groupby("component")["amp_ratio"].median()
all_q = float(fit["amp_ratio"].median())
print("median ratio per component:", median_q.round(3).to_dict(), "; all rows:", round(all_q, 3))
print("Delta Mw = (2/3) log10(median ratio) per component:",
      {comp: round(float(2.0 / 3.0 * np.log10(q)), 3) for comp, q in median_q.items()},
      "; all rows:", round(float(2.0 / 3.0 * np.log10(all_q)), 3))
print("moment of Mw", KIGAM_MW, "+- 0.05 against Mw", KIGAM_MW, ": factor",
      round(float(magnitude_moment(mw=KIGAM_MW - 0.05) / m0_used), 3), "to",
      round(float(magnitude_moment(mw=KIGAM_MW + 0.05) / m0_used), 3))

The two R3 runs: the fingerprints of the first run and of its unchanged rerun, and your table for both velocity
models side by side.

In [ ]:
print("SHA-256 of result.npz:")
for label, folder in (("first run", FK_MAIN), ("unchanged rerun", FK_AGAIN), (CHANGED_MODEL, FK_CHANGED)):
    print(f"  {label:16s} {sha256(folder / 'result.npz')}")
print("unchanged rerun: identical result.npz:", sha256(FK_MAIN / "result.npz") == sha256(FK_AGAIN / "result.npz"),
      "; largest difference of the samples:",
      float(max(np.abs(a - b).max() for a, b in ((sangju.z, again.z), (sangju.r, again.r), (sangju.t, again.t)))))
both = pd.DataFrame(fit_rows + changed_rows)
compare = both.pivot_table(index=["station", "component"], columns="run", values=["cc", "lag_s", "amp_ratio"])
print(compare.round(3).to_string())
print("first P arrival (s) per station,", VELOCITY_MODEL, np.round(sangju.t_p_s, 2), CHANGED_MODEL,
      np.round(changed.t_p_s, 2))
print("first S arrival (s) per station,", VELOCITY_MODEL, np.round(sangju.t_s_s, 2), CHANGED_MODEL,
      np.round(changed.t_s_s, 2))
for name in (VELOCITY_MODEL, CHANGED_MODEL):
    layer = st.data.velocity_model(name)
    at_source = layer[layer["top_km"] <= KIGAM_DEPTH_KM].iloc[-1]
    print(f"{name}: at the source ({KIGAM_DEPTH_KM} km) Vp {at_source['vp']} km/s, Vs {at_source['vs']} km/s; "
          f"top layer Vp {layer['vp'].iloc[0]}, Vs {layer['vs'].iloc[0]} km/s, down to "
          f"{layer['top_km'].iloc[1]:g} km")
print("median per run:\n", both.groupby("run")[["cc", "lag_s", "amp_ratio"]].median().round(3).to_string())

**Explain (3-5 sentences):** Using your printed table: on which stations and components do synthetic and record
agree in shape, and how close are the amplitudes? What happens at the station near the nodal plane, and why? Read
your printed Delta Mw: would a moment smaller (or larger) by that amount explain the ratios, and which other causes
can your numbers not rule out? What did the unchanged rerun show? When you changed the velocity model, which
numbers moved (lags, correlations, ratios) and in which direction, and what in the two models can explain that? If
a number moved in a way you cannot explain from what the notebook printed, say so plainly.

## What you did and what comes next

You built a moment tensor from a fault, put it on the magnitude scale, predicted its radiation, wrote a far-field
seismogram and checked it against an FK program, found out what that program wants and returns, and compared
layered-Earth synthetics of the Sangju earthquake with its records. Module M53 turns this around: from records
and Green's functions back to the moment tensor.

Credits: KIGAM mechanism from KIGAM (2021), report KEMS-2019-062; KMA catalog (KOGL Type 1); station metadata
StationMetadata by Byeong-Seok Ahn (PKNU; Ahn et al. 2024); PyFK by Xi et al. (https://github.com/ziyixi/pyfk), the
FK method of Zhu and Rivera (2002); velocity models Kim and Kim (1983) and Kim et al. (2011).